# ASR v4 — Whisper Small LoRA r32 trên Colab GPU
Train, validation và test đều chạy trên Colab GPU. Train chỉ dùng train v4; checkpoint/early stopping chỉ dùng validation WER rồi CER; model được khóa trước khi test. V4 tái sử dụng đúng ba split v3 nên đây là so sánh cùng test set, không phải holdout mới. Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
from pathlib import Path
import csv, json, os, shutil, subprocess, sys
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount')
REPO_URL = 'https://github.com/TiiAyyLuvBear/VoiceStudy-Assistant.git'
BRANCH = 'master'  # branch mặc định hiện tại của repository
RUN_NAME = 'whisper-small-lora-wide-v4-run1'  # đổi nếu chạy thí nghiệm mới
PROJECT_ROOT = Path('/content/VoiceStudy-Assistant')
RUN_ROOT = Path('/content/drive/MyDrive/VoiceStudy-Assistant-Colab') / RUN_NAME
MODEL_ROOT = RUN_ROOT / 'models/experimental/asr/v4'
REPORT_ROOT = RUN_ROOT / 'reports/asr/v4'
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
elif PROJECT_ROOT.exists():
    raise RuntimeError(f'{PROJECT_ROOT} tồn tại nhưng không phải Git repo')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Output sẽ lưu tại:', RUN_ROOT)

In [ ]:
# Preflight GPU, dữ liệu v4 và kết quả tham chiếu original/v3.
import torch
from src.utils import canonical_csv_sha256
assert torch.cuda.is_available(), 'Hãy bật T4 GPU trong Runtime settings'
print('GPU:', torch.cuda.get_device_name(0))
manifest_path = Path('data/processed/v4/asr_finetune_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['dataset_version'] == 'v4' and manifest['freeze_status'] == 'DEVELOPMENT'
for split, expected in {'train': 1465, 'validation': 322, 'test': 249}.items():
    path = Path(f'data/processed/v4/metadata/asr_finetune_{split}.csv')
    with path.open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream))
    missing = [row['audio_path'] for row in rows if not Path(row['audio_path']).is_file()]
    assert len(rows) == expected and not missing, (split, len(rows), len(missing))
    assert canonical_csv_sha256(path) == manifest['datasets'][split]['canonical_csv_sha256']
    print(split, len(rows), 'rows — OK')
for path in [Path('reports/asr/v3/baseline_original/baseline_original_metrics.json'), Path('reports/asr/v3/final_test/test_metrics.json')]:
    assert path.is_file(), f'Thiếu file cần push lên Git: {path}'
from huggingface_hub import snapshot_download
BASE_MODEL = Path('/content/models/openai-whisper-small')
snapshot_download(repo_id='openai/whisper-small', local_dir=BASE_MODEL)
print('Base model:', BASE_MODEL)

In [ ]:
# TRAIN + VALIDATION: r32; q/k/v/out + fc1/fc2; 4–6 epoch.
# Batch 4 + accumulation 4 cho effective batch 16 trên T4 16 GB.
training_summary = MODEL_ROOT / 'training_summary.json'
if training_summary.exists():
    print('Đã train xong, bỏ qua:', training_summary)
elif (MODEL_ROOT / 'best_adapter').exists():
    raise RuntimeError('Run bị gián đoạn; đổi RUN_NAME để chạy sạch, không ghi đè checkpoint')
else:
    subprocess.run([
        sys.executable, '-m', 'scripts.finetune_asr_v4',
        '--base-model', str(BASE_MODEL), '--output-root', str(MODEL_ROOT),
        '--device', 'cuda', '--mixed-precision', 'fp16',
        '--batch-size', '4', '--gradient-accumulation', '4',
        '--validation-batch-size', '8',
        '--min-epochs', '4', '--max-epochs', '6',
        '--early-stopping-patience', '1'
    ], check=True)

In [ ]:
# Export checkpoint validation tốt nhất, convert CTranslate2, rồi KHÓA trước test.
merged_dir = MODEL_ROOT / 'hf_merged'
ct2_dir = MODEL_ROOT / 'ctranslate2'
lock_path = MODEL_ROOT / 'locked_model.json'
if not merged_dir.exists():
    subprocess.run([sys.executable, '-m', 'scripts.export_asr_v4',
        '--base-model', str(BASE_MODEL), '--adapter', str(MODEL_ROOT / 'best_adapter'),
        '--training-summary', str(training_summary), '--output-dir', str(merged_dir)], check=True)
if not ct2_dir.exists():
    subprocess.run(['ct2-transformers-converter', '--model', str(merged_dir),
        '--output_dir', str(ct2_dir), '--quantization', 'float16'], check=True)
if not lock_path.exists():
    subprocess.run([sys.executable, '-m', 'scripts.lock_asr_v4_model',
        '--model-dir', str(ct2_dir), '--training-summary', str(training_summary),
        '--export-summary', str(MODEL_ROOT / 'export_summary.json'), '--output', str(lock_path),
        '--device', 'cuda', '--compute-type', 'float16'], check=True)
print('Locked:', lock_path)

In [ ]:
# TEST v4 đúng một lần sau khi lock.
test_dir = REPORT_ROOT / 'final_test'
test_metrics = test_dir / 'test_metrics.json'
if test_metrics.exists():
    print('Test v4 đã tồn tại; không chạy lại:', test_metrics)
else:
    subprocess.run([sys.executable, '-m', 'scripts.evaluate_faster_whisper_v4',
        '--model-dir', str(ct2_dir), '--lock', str(lock_path),
        '--output-dir', str(test_dir), '--confirm-test'], check=True)


In [ ]:
# So sánh ba model, freeze manifest và in WER/CER.
comparison_path = REPORT_ROOT / 'comparison.json'
final_manifest = MODEL_ROOT / 'final_manifest.json'
if not comparison_path.exists():
    subprocess.run([sys.executable, '-m', 'scripts.finalize_asr_v4',
        '--v4', str(test_metrics), '--lock', str(lock_path),
        '--training-summary', str(training_summary),
        '--comparison', str(comparison_path), '--final-manifest', str(final_manifest)], check=True)
shutil.copy2('data/processed/v4/asr_finetune_manifest.json', RUN_ROOT / 'asr_finetune_manifest_frozen.json')
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
print('Kết quả cuối:', comparison_path)
for model in comparison['models']:
    print(model['label'], 'WER={:.2f}%'.format(model['wer'] * 100), 'CER={:.2f}%'.format(model['cer'] * 100))